In [ ]:
"""
Normality tests for copiesperml (qPCR) and rpkmf (sequencing), per virus.

Data:   Data_3_Correlation_Data_Final.xlsx (one sheet per virus)
Filter: mismatch_flag_after_correction == 'none', then each variable is
        restricted to its own detected subset (> 0). Zeros are structural
        non-detects, not measured values, so they are excluded.
Scale:  raw values only (no log transform), matching the downstream analysis.

Tests:
  - Shapiro-Wilk: most powerful general normality test; p-values reliable to n=5000.
  - Kolmogorov-Smirnov with Lilliefors correction: KS where the normal's mean/SD
    are estimated from the sample (plain scipy kstest is invalid in that case).
    statsmodels caps reported p-values to [0.001, 0.2]; p=0.001 means <= 0.001.
  - Anderson-Darling: tail-weighted; reported as statistic vs 5% critical value.
  - Skewness and excess kurtosis (0 for a normal) as effect-size descriptors.

Output: normality_tests_copiesperml_rpkmf_nodallas.xlsx (one row per virus/variable)
"""
import pandas as pd
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors

INPUT = "Data_3_Correlation_Data_Final.xlsx"
OUTPUT = "Supp_Table_1_normality_tests_copiesperml_rpkmf_nodallas.xlsx"
VIRUSES = ["fluA", "fluB", "rsvA", "rsvB", "sars-cov-2"]
VARIABLES = ["copiesperml", "rpkmf"]
ALPHA = 0.05

rows = []
for virus in VIRUSES:
    d = pd.read_excel(INPUT, sheet_name=virus)
    n_sheet = len(d)
    d = d[d["mismatch_flag_after_correction"] == "none"]
    for var in VARIABLES:
        x = d.loc[d[var] > 0, var].dropna().to_numpy()
        sw = stats.shapiro(x)
        rows.append({
            "virus": virus,
            "variable": var,
            "n_sheet": n_sheet,
            "n_after_mismatch_filter": len(d),
            "n_tested": len(x),
            "shapiro_W": sw.statistic,
            "shapiro_p": sw.pvalue,
            "skew": stats.skew(x),
            "excess_kurtosis": stats.kurtosis(x),
            "shapiro_reject": sw.pvalue < ALPHA,
        })

res = pd.DataFrame(rows)
res.to_excel(OUTPUT, index=False)

pd.set_option("display.width", 200)
for var in VARIABLES:
    sub = res[res.variable == var]
    print(f"\n=== {var} (raw) ===")
    print(sub[["virus", "n_tested", "shapiro_W", "shapiro_p","skew", "excess_kurtosis"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\nSaved {OUTPUT}")



=== copiesperml (raw) ===
     virus  n_tested  shapiro_W  shapiro_p  skew  excess_kurtosis
      fluA       342     0.0877   2.22e-37  11.2              131
      fluB       175     0.0534   1.12e-28  13.1              170
      rsvA       883     0.0132   5.68e-55  29.7              878
      rsvB       913      0.122   1.14e-53  14.8              239
sars-cov-2      3574      0.359   2.27e-77  9.79              149

=== rpkmf (raw) ===
     virus  n_tested  shapiro_W  shapiro_p  skew  excess_kurtosis
      fluA      1400      0.327   1.31e-57  13.6              295
      fluB       663      0.158   4.34e-47  20.9              492
      rsvA       963      0.565   1.27e-43  5.13             41.4
      rsvB       347      0.583   5.11e-28  4.69             33.3
sars-cov-2      3209      0.482   1.67e-70   8.4              132

Saved Supp_Table_1_normality_tests_copiesperml_rpkmf_nodallas.xlsx
